# Homework #7

## Disorder

### Simplest disorder model: 1D Anderson model

#### One dimensional time-independent Schrodinger equation with a disorded potencial

Consider a continuous model of a 1D particle in a disordered potencial $V(x)$. By discreting the Schrodinger equation on a lattice with sufficently small spacing, much smaller than the de Broglie wavelength, we can obtain an approximation of the Anderson model.

Starting with the Schrodinger equation:

$$
- \frac{\hbar^2}{2m_0} \frac{d^2 \psi}{dx^2} + V(x) \psi = E_i \psi
$$

where $V(x)$ is some random potencial and $E_i$ is the eigen energy.

#### Discretization

Discretizing the equation for small, but finite, differences allow the second order derative to be expressed as 

$$
\frac{d^2 \psi}{dx^2} \approx \frac{\psi(x + \Delta x) + \psi(x - \Delta x) - 2\psi(x)}{\Delta x^2}
$$

So the $\frac{d^2}{dx^2}$ operator can be expressed as a tridiagonal matrix (zero boundary condition)

$$
\begin{bmatrix}
-2 & 1 & 0 & &  \\
1 & \ddots & \ddots & \ddots & \\
0 & \ddots & \ddots & \ddots & 0 \\
& \ddots & \ddots & \ddots & 1 \\
& & 0 & 1 & -2
\end{bmatrix}
$$

and the potencial operator is a diagonal matrix

$$
\begin{bmatrix}
V(x_1) & 0 & 0 & \cdots & 0 \\
0 & V(x_2) & 0 & \cdots & 0 \\
0 & 0 & V(x_3) & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & \cdots & V(x_N)
\end{bmatrix}
$$

so the equation in matrix form is

(matrix)

where $a_0 = \frac{\hbar^2}{2m(\Delta x)^2}$

#### 1D Anderson Model

The previous discretization motivates the Anderson model, whose 1D Hamiltonian is 

$$
H = \sum_{n=-\infty}^{\infty} \left( w_n |n\rangle \langle n| + t |n\rangle \langle n+1| + t |n+1\rangle \langle n| \right)
$$

where the state $|n\rangle$ is the occupation amplitude of the site $n$ with $w_n$ its on-site energy. $t$ is the tunneling matrix element coupling neighb ouring sites, traditionally taken with the value $t = 1$.

Disorder is introduced by allowing the on-site energies $w_n$ to be random variables. The standart choice is to take $w_n$ to be uncorrelated random variables uniformly distributed in the interval $[-W/2, W/2]$, with $W$ measuring the disorder strength.

#### The Inverse Participation Ratio

In order to decide whether or not a state is localized it's often sufficent to consider the second moment of the probability density,

$$
\text{IPR} = P^{-1} = \sum_{r} |\psi(r)|^4 = N \cdot \left( \frac{1}{N} \right)^2 = \frac{1}{N}
$$

This is the inverse participation number . It's a measure of the portion of the space where the amplitude of the wavefunction differs from zero. IPR vs energy plots reveal critical energies where the states transition from extended to localized.

#### Level-Spacing Distribution

If we have an $N \times N$ Hamiltonian matrix for a 3D sample of length $L$, we can calculate $N = L^3$ energies. 

### Calculate the density states, IPR of each eigenstate vs energy, and level spacings of the 1D Anderson model for several values of disorder W.

In [ ]:
ModelAnalysis[n_, W_] := Module[
    {
        (* Start variables*)
        w,
        hamiltonian,
        eigenValues, eigenVectors,
        normalizedEigenVectors, IPR,
        levelSpacings
    },
    
    (* Random energy values *)
    w = RandomReal[{-W/2, W/2}, n];
    
    (* Construct hamiltonian matrix *)
    hamiltonian = DiagonalMatrix[w] + 
                 DiagonalMatrix[ConstantArray[-1, n-1], 1] + 
                 DiagonalMatrix[ConstantArray[-1, n-1], -1];
    
    (* Compute hamiltonian's eigenvalues and eigenvectors*)
    {eigenValues, eigenVectors} = Eigensystem[hamiltonian];
    
    (* Normalize eigenvectors and calculate IPR *)
    normalizedEigenVectors = Normalize /@ eigenVectors;
    IPR = Total[#^4] & /@ normalizedEigenVectors;
    
    (* Calculate level spacings *)
    levelSpacings = Differences[Sort[eigenValues]];
    
    (* Return separate plots *)
    <|
        "DOS" -> ListPlot[
            Tally[Round[eigenValues, 0.1]], 
            PlotLabel -> StringForm["Density of States (W = `1`)", W],
            AxesLabel -> {"Energy", "Density"},
            ImageSize -> Medium
        ],
        
        "IPR" -> ListPlot[
            Transpose[{eigenValues, IPR}],
            PlotLabel -> StringForm["IPR vs Energy (W = `1`)", W],
            AxesLabel -> {"Energy", "IPR"},
            PlotRange -> All,
            ImageSize -> Medium
        ],
        
        "LevelSpacing" -> Histogram[
    levelSpacings,
    {Min[levelSpacings], Max[levelSpacings], (Max[levelSpacings]-Min[levelSpacings])/100},
    (* Creates 100 equally spaced bins across the full range *)
    "PDF",
    
    BarOrigin -> Bottom,
    PlotLabel -> StringForm["Level Spacing Distribution (W = `1`)", W],
    AxesLabel -> {"Level Spacing", "Frequency"},
    ImageSize -> Medium
]
    |>
]

(* Parameters *)
n = 1000; (* Dimension of the matrix *)
t = 1;  (* Hopping parameter *)

We can expect to see differing results for each of the plots, depending on the value of our disorder magnitude $W$:

In [ ]:
(* No disorder -> W = 0 *)
W = 0; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

For a theoritical case, where W = 0, the DoS follows a curve similar to what is seen on the tight-binding model. Physically, this tells us all states are extended and delocalized. This is also supported by the IPR, which just shows a flat curve, meaning all states contribute equally. The LSD takes on a Wigner-Dyson distribution, with the extended states far apart from each other, possibly due to them repelling each other. This behaviour is in line with what is expected of a metallic, conducting system.

Although, this is contradictory because extended states can not exist in 1D.

In [ ]:
(* Small disorder -> W = 0.01 *)
W = 0.01; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

By introducing as small amount of disorder, while the DoS looks largely the same, we can see spikes on the outer edges of the IPR. Those spikes indicate localized states, while the remaining states are **quasi-extended***. As for the LSD, while it still maintains the Wigner-Dyson distribution, the peak is moving to the left, as the distance between states decreases.

*they appear extended for a finite system, but would localize in the thermodynamic limit, $N -> \infty$.

In [ ]:
(* Low disorder -> W = 0.15 *)
W = 0.15; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

Here we see the DoS is now beginning to break away from the clean tigh-binding curve, and the spikes on the IPR are more pronounced. The peak of the LSD again continues moving to the left.

In [ ]:
(* Mild disorder -> W = 0.5 *)
W = 0.5; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

The tails on the DoS can now be seen more easily, again supporting the existance of an ever-increasing number of localized states. The IPR and LSD follow previously established patterns.

In [ ]:
(* Moderate disorder -> W = 1 *)
W = 1; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

The DoS is losing it's resemblance to the tight-binding model curve, indicating weak loaclization for lower energies, while the edges states get more localized, increasing the width of the curve's tails. The IDR follows a similar behaviour. Essentially all states are now localized, albeit with different levels of localization. For W = 1, we see the crossover point from a Wigner-Dyson distribution. All LSD graphs going forward should resemble a Poisson distribution curve.

In [ ]:
(* Strong disorder -> W = 5 *)
W = 5; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

The DoS no longer bears any sort of resemblance to the tight-binding curve, as even the lower energy states get increasingly more localized. As expected, the LSD now resembles a Poisson distribution curve, implying at the states are now much closer together.

In [ ]:
(* Extreme disorder -> W = 10 *)
W = 100; 

(* Generate analysis *)
analysis = ModelAnalysis[n, W];

GraphicsRow[{
    analysis["DOS"],
    analysis["IPR"]
}, ImageSize -> Large]

analysis["LevelSpacing"]

Interestingly, for extreme levels of disorder, the now strongly localized states seem to all converge on discrete levels of density. This results in a flat curve with insanely wide tales. Additionally, for most states, their IPR now tends to 1. Most states are now equally localized.